In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

# Read from your managed Delta table
df_model = spark.sql("SELECT * FROM sensor_telemetry")

# Convert to Pandas if you plan to use Scikit-Learn/XGBoost locally in the notebook
pdf = df_model.toPandas()
pdf.head()

from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = pdf[['temperature', 'vibration']]
y = pdf['failure_status']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Initialize and train the model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

StatementMeta(, 76da7ea9-ed65-4899-bf9a-9b81725530b2, 13, Finished, Available, Finished, False)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



In [4]:
import pandas as pd

# 1. Run inference on your dataset using your trained model
pdf['predicted_failure'] = model.predict(X)
pdf['failure_probability'] = model.predict_proba(X)[:, 1] # Probability of failure

# 2. Convert the Pandas dataframe back to a Spark dataframe
df_predictions = spark.createDataFrame(pdf)

# 3. Save the results as a new managed Delta table for downstream consumption
df_predictions.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sensor_predictions")

print("Successfully generated predictions and saved to 'sensor_predictions' Delta table!")

StatementMeta(, 76da7ea9-ed65-4899-bf9a-9b81725530b2, 22, Finished, Available, Finished, False)

Successfully generated predictions and saved to 'sensor_predictions' Delta table!
